
# Metallicity Distribution Functions observadas

Notebook para descargar los siguientes catálogos de Vizier:
- **Haywood (2001)**: distribución de metalicidades de enanas del vecindario solar
- **Geneva–Copenhagen Survey III, Holmberg et al. (2009)**: estrellas F y G del vecindario solar
- **Adibekyan et al. (2012)**: 1111 estrellas FGK con abundancias detalladas, incluyendo $[\mathrm{Fe/H}]$ y $[\mathrm{Mg/H}]$

El objetivo es construir distribuciones de metalicidad observadas (MDF) y comparar las MDF obtenidas con distintas muestras estelares de su simulación 



## Metalicidad del modelo y abundancias observadas

El modelo de evolución química de caja cerrada sigue la metalicidad total del gas

$$
Z = \frac{M_{\rm metals}}{M_{\rm gas}}
$$

Para compararla con observaciones es útil escribirla en escala logarítmica respecto del Sol

$$
[\mathrm{M/H}]
=
\log_{10}\left(\frac{Z}{Z_\odot}\right)
$$

Los catálogos observacionales normalmente no miden directamente $Z$. En cambio, entregan abundancias de elementos individuales como $[\mathrm{Fe/H}]$ o $[\mathrm{Mg/H}]$

Por lo tanto, al comparar el modelo con los datos debemos decidir qué abundancia observacional utilizaremos como **trazador de la metalicidad global**.


In [ ]:
# !pip -q install astroquery


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astroquery.vizier import Vizier



## Adibekyan et al. (2012)

El catálogo `J/A+A/545/A32/table4` contiene abundancias químicas para 1111 estrellas FGK.

Descargaremos directamente $[\mathrm{Fe/H}]$ y $[\mathrm{Mg/H}]$.


In [ ]:
vizier_adi = Vizier(
    columns=["Star", "[Fe/H]", "[Mg/H]"],
    row_limit=-1 #para que no tenga limite de filas
)

adi = vizier_adi.get_catalogs("J/A+A/545/A32/table45")[0]

print(len(adi))
print(adi.colnames)
adi[:5]


In [ ]:
#eliminamos NANs
feh_adi = np.array(adi["[Fe/H]"].filled(np.nan), dtype=float)
mgh_adi = np.array(adi["[Mg/H]"].filled(np.nan), dtype=float)

feh_adi = feh_adi[np.isfinite(feh_adi)]
mgh_adi = mgh_adi[np.isfinite(mgh_adi)]

print("Fe/H válidos:", len(feh_adi))
print("Mg/H válidos:", len(mgh_adi))



#### Compare la distribución de $[\mathrm{Fe/H}]$ y $[\mathrm{Mg/H}]$ (visualizacion de histogramas)



- ¿Tienen $[\mathrm{Fe/H}]$ y $[\mathrm{Mg/H}]$ exactamente la misma distribución?
- ¿Dónde aparecen las principales diferencias?
- ¿Qué podría hacer que Fe y Mg no sigan exactamente la misma evolución química?



## Haywood (2001)

El catálogo `J/MNRAS/325/1365/table3` contiene 348 estrellas.

La columna `ST` identifica con `*` las **selected long-lived dwarf stars**. Para construir la MDF utilizaremos solamente esa submuestra.


In [ ]:

vizier_hay = Vizier(
    columns=["HIP", "[Fe/H]", "ST"],
    row_limit=-1
)

hay = vizier_hay.get_catalogs("J/MNRAS/325/1365/table3")[0]

print(len(hay))
print(hay.colnames)
hay[:5]


In [ ]:
seleccion = np.array(hay["ST"].filled(""), dtype=str) == "*"

hay_long = hay[seleccion]

feh_hay = np.array(hay_long["[Fe/H]"].filled(np.nan), dtype=float)
feh_hay = feh_hay[np.isfinite(feh_hay)]

print("Long-lived dwarfs:", len(hay_long))
print("Fe/H válidos:", len(feh_hay))



## Geneva–Copenhagen Survey III

El catálogo `V/130/gcs3` contiene 16682 estrellas F y G del vecindario solar.

Usaremos su estimación de $[\mathrm{Fe/H}]$.


In [ ]:

vizier_gcs = Vizier(
    columns=["HIP", "[Fe/H]"],
    row_limit=-1
)

gcs = vizier_gcs.get_catalogs("V/130/gcs3")[0]

print(len(gcs))
print(gcs.colnames)
gcs[:5]


In [ ]:
feh_gcs = np.array(gcs["[Fe/H]"].filled(np.nan), dtype=float)
feh_gcs = feh_gcs[np.isfinite(feh_gcs)]

print("Fe/H válidos:", len(feh_gcs))



#### Visualización: compare las 3 MDF en $[\mathrm{Fe/H}]$

Normalice los histogramas, ya que la cantidad de estrellas es distinta entre catálogos



### Importante

Las tres MDF no corresponden exactamente a la misma selección de estrellas. Los catálogos difieren en tamaño de muestra, criterios de selección y técnicas utilizadas para estimar las abundancias.

Diferencias entre las curvas no deben interpretarse automáticamente como diferencias físicas en la evolución química de la Galaxia.



### $[\mathrm{Mg/H}]$ o $[\mathrm{Fe/H}]$  como proxy de $Z$

El modelo predice la cantidad de metales $Z$, mientras que las observaciones miden elementos individuales.

Supongamos que una estrella tiene aproximadamente la mitad de la metalicidad solar

$$
\frac{Z}{Z_\odot}=0.5
$$

Si Mg representa aproximadamente la misma fracción de los metales que en el Sol, entonces también esperamos aproximadamente la mitad de la abundancia solar de Mg

$$
\frac{(\mathrm{Mg/H})}{(\mathrm{Mg/H})_\odot}\approx0.5
$$

Por lo tanto

$$
[\mathrm{M/H}]
=
\log_{10}(0.5)
\approx
[\mathrm{Mg/H}]
$$

La idea no es que **Mg sea la metalicidad total**, sino que su abundancia puede utilizarse como indicador del nivel general de enriquecimiento químico.

Esta es una aproximación y debe justificarse.


#### Mg versus Fe

- **Mg**: Mg es un elemento $\alpha$ producido principalmente por estrellas masivas y liberado rápidamente al medio interestelar por supernovas de colapso del núcleo. 

- **Fe**: Fe también es un trazador muy utilizado de metalicidad. Sin embargo, una parte importante del Fe proviene de supernovas Ia, que enriquecen el medio con un retardo temporal.

Para comparar la MDF observada con la predicción del modelo deberán escoger uno de estos elementos como trazador de la metalicidad.
La elección debe ser justificada considerando:

- el origen del elemento
- su escala temporal de enriquecimiento
- las hipótesis del closed-box model
- las limitaciones de utilizar un elemento individual como proxy de $Z$


In [ ]:

# Esta celda se utilizará más adelante, cuando tengan Z_birth

# Z_sun = 0.014
# Z_birth = ...

# mh_model = np.log10(Z_birth / Z_sun)

# proxy_obs = mgh_adi    # si el grupo escoge Mg
# proxy_obs = feh_adi    # si el grupo escoge Fe

# bins = np.arange(-2.5, 0.7, 0.1)

# plt.figure(figsize=(8, 5))
# plt.hist(mh_model, bins=bins, density=True,
#          histtype="step", linewidth=2, label="Closed-box model")
# plt.hist(proxy_obs, bins=bins, density=True,
#          histtype="step", linewidth=2, label="Observaciones")

# plt.xlabel("[M/H] modelo / [X/H] observado")
# plt.ylabel("Densidad de probabilidad")
# plt.legend()
# plt.show()



## Referencias de catálogos

- Adibekyan, V. Zh. et al. 2012, *A&A*, **545**, A32
- Haywood, M. 2001, *MNRAS*, **325**, 1365
- Holmberg, J., Nordström, B. & Andersen, J. 2009, *A&A*, **501**, 941
- Bovy, J., *Dynamics and Astrophysics of Galaxies*, capítulo de evolución química galáctica
